In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# 1. Define paths and forecasting settings
input_path = Path(
    "results/step_04/engineered_panel.csv"
)

output_folder = Path(
    "results/step_05"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

ENCODER_LENGTH = 12
FORECAST_HORIZON = 6

TRAIN_END_DATE = pd.Timestamp("2023-12-01")
VALIDATION_ORIGIN_DATE = pd.Timestamp("2023-12-01")
TEST_ORIGIN_DATE = pd.Timestamp("2024-06-01")

feature_names = [
    "Quantity",
    "Unit_Margin",
    "Total_Purchase_Cost",
    "Fuzzy_TOPSIS_Score",
    "time_idx",
    "month_sin",
    "month_cos"
]

log_feature_names = [
    "Quantity",
    "Unit_Margin",
    "Total_Purchase_Cost"
]


# 2. Load the engineered panel
panel = pd.read_csv(
    input_path,
    parse_dates=["Entry_Date"]
)

required_columns = (
    feature_names
    + ["Reference", "Entry_Date", "Subsystem"]
)

missing_columns = (
    set(required_columns) - set(panel.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if panel.duplicated(
    ["Reference", "Entry_Date"]
).any():
    raise ValueError(
        "Duplicate part-month observations were found."
    )

panel = panel.sort_values(
    ["Entry_Date", "Reference"]
).reset_index(drop=True)


# 3. Define a consistent ordering of dates and parts
dates = pd.DatetimeIndex(
    sorted(panel["Entry_Date"].unique())
)

references = sorted(
    panel["Reference"].unique()
)

n_months = len(dates)
n_parts = len(references)
n_features = len(feature_names)

expected_dates = pd.date_range(
    start=dates.min(),
    end=dates.max(),
    freq="MS"
)

if not dates.equals(expected_dates):
    raise ValueError(
        "The panel must contain consecutive month-start dates."
    )

# Verify that every month contains every part.
expected_index = pd.MultiIndex.from_product(
    [dates, references],
    names=["Entry_Date", "Reference"]
)

indexed_panel = panel.set_index(
    ["Entry_Date", "Reference"]
)

if (
    len(indexed_panel) != len(expected_index)
    or not expected_index.isin(indexed_panel.index).all()
):
    raise ValueError(
        "A complete part-month panel is required."
    )

# Reindex explicitly to avoid relying on implicit row ordering.
panel = (
    indexed_panel
    .reindex(expected_index)
    .reset_index()
)

# Verify the time index created in Code 4.
expected_time_index = np.repeat(
    np.arange(n_months),
    n_parts
)

np.testing.assert_array_equal(
    panel["time_idx"].to_numpy(),
    expected_time_index
)


# 4. Identify the training cutoff and forecast origins
def get_date_index(date):

    if date not in dates:
        raise ValueError(
            f"Required date not found: {date.date()}"
        )

    return int(dates.get_loc(date))


train_end_index = get_date_index(
    TRAIN_END_DATE
)

validation_origin_index = get_date_index(
    VALIDATION_ORIGIN_DATE
)

test_origin_index = get_date_index(
    TEST_ORIGIN_DATE
)

if test_origin_index + FORECAST_HORIZON >= n_months:
    raise ValueError(
        "Insufficient observations for the six-month test horizon."
    )


# 5. Convert the panel into time-by-part arrays
raw_features = (
    panel[feature_names]
    .to_numpy(dtype=np.float64)
    .reshape(n_months, n_parts, n_features)
)

raw_targets = (
    panel["Quantity"]
    .to_numpy(dtype=np.float64)
    .reshape(n_months, n_parts)
)

if not np.isfinite(raw_features).all():
    raise ValueError(
        "The feature matrix contains missing or infinite values."
    )

if (raw_targets < 0).any():
    raise ValueError(
        "Demand quantities must be nonnegative."
    )


# 6. Apply log1p to historical quantity and financial features
transformed_features = raw_features.copy()

for feature in log_feature_names:

    feature_index = feature_names.index(feature)

    if (
        transformed_features[:, :, feature_index] < 0
    ).any():
        raise ValueError(
            f"Negative values found in log-transformed feature: {feature}"
        )

    transformed_features[:, :, feature_index] = np.log1p(
        transformed_features[:, :, feature_index]
    )


# 7. Fit the feature scaler on training observations only
training_feature_rows = (
    transformed_features[:train_end_index + 1]
    .reshape(-1, n_features)
)

feature_scaler = StandardScaler()

feature_scaler.fit(
    training_feature_rows
)

scaled_features = (
    feature_scaler
    .transform(
        transformed_features.reshape(-1, n_features)
    )
    .reshape(n_months, n_parts, n_features)
    .astype(np.float32)
)


# 8. Calculate the target scaling factor from training data only
target_scale = max(
    float(
        raw_targets[:train_end_index + 1].std()
    ),
    1.0
)

scaled_targets = (
    raw_targets / target_scale
).astype(np.float32)


# 9. Define a function for constructing historical windows
def build_windows(origins):

    origins = np.asarray(
        origins,
        dtype=int
    )

    x_windows = []
    y_windows = []
    y_raw_windows = []

    for origin in origins:

        encoder_start = (
            origin - ENCODER_LENGTH + 1
        )

        decoder_end = (
            origin + FORECAST_HORIZON
        )

        if encoder_start < 0:
            raise ValueError(
                "Insufficient history for an encoder window."
            )

        if decoder_end >= n_months:
            raise ValueError(
                "Insufficient observations for a target window."
            )

        # Input shape:
        # [historical months, parts, features]
        x_window = scaled_features[
            encoder_start:origin + 1
        ]

        # Target shape:
        # [parts, future horizons]
        y_window = scaled_targets[
            origin + 1:decoder_end + 1
        ].T

        y_raw_window = raw_targets[
            origin + 1:decoder_end + 1
        ].T

        x_windows.append(
            x_window.copy()
        )

        y_windows.append(
            y_window.copy()
        )

        y_raw_windows.append(
            y_raw_window.copy()
        )

    return (
        np.stack(x_windows).astype(np.float32),
        np.stack(y_windows).astype(np.float32),
        np.stack(y_raw_windows)
    )


# 10. Define the training origins
# Every training target must end by the training cutoff.
training_origins = np.arange(
    ENCODER_LENGTH - 1,
    train_end_index - FORECAST_HORIZON + 1
)

if len(training_origins) == 0:
    raise ValueError(
        "No complete training windows are available."
    )

validation_origins = np.array(
    [validation_origin_index]
)

test_origins = np.array(
    [test_origin_index]
)


# 11. Construct training, validation, and test arrays
X_train, y_train, y_train_raw = build_windows(
    training_origins
)

X_val, y_val, y_val_raw = build_windows(
    validation_origins
)

X_test, y_test, y_test_raw = build_windows(
    test_origins
)


# 12. Verify temporal separation and array dimensions
assert (
    training_origins + FORECAST_HORIZON
    <= train_end_index
).all()

assert (
    validation_origin_index + FORECAST_HORIZON
    <= test_origin_index
)

assert X_train.shape[1:] == (
    ENCODER_LENGTH,
    n_parts,
    n_features
)

assert y_train.shape[1:] == (
    n_parts,
    FORECAST_HORIZON
)

for array in [
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test
]:
    assert np.isfinite(array).all()

# Verify that target scaling can be reversed.
np.testing.assert_allclose(
    y_test * target_scale,
    y_test_raw,
    rtol=1e-5,
    atol=1e-5
)


# 13. Record the forecast origins and target periods
def describe_windows(origins, partition):

    records = []

    for origin in origins:

        records.append({
            "Partition": partition,
            "Origin_Index": int(origin),
            "Encoder_Start": dates[
                origin - ENCODER_LENGTH + 1
            ].strftime("%Y-%m-%d"),
            "Origin_Date": dates[
                origin
            ].strftime("%Y-%m-%d"),
            "First_Target_Date": dates[
                origin + 1
            ].strftime("%Y-%m-%d"),
            "Last_Target_Date": dates[
                origin + FORECAST_HORIZON
            ].strftime("%Y-%m-%d")
        })

    return records


window_records = (
    describe_windows(training_origins, "Training")
    + describe_windows(validation_origins, "Validation")
    + describe_windows(test_origins, "Test")
)

window_table = pd.DataFrame(
    window_records
)

window_table.to_csv(
    output_folder / "forecast_windows.csv",
    index=False
)


# 14. Save the common test target identities
test_target_records = []

for part_index, reference in enumerate(references):

    for horizon in range(1, FORECAST_HORIZON + 1):

        test_target_records.append({
            "Reference": reference,
            "Origin_Date": TEST_ORIGIN_DATE.strftime(
                "%Y-%m-%d"
            ),
            "Target_Date": dates[
                test_origin_index + horizon
            ].strftime("%Y-%m-%d"),
            "Horizon": horizon,
            "Actual": float(
                y_test_raw[
                    0,
                    part_index,
                    horizon - 1
                ]
            )
        })

test_target_table = pd.DataFrame(
    test_target_records
)

test_target_table.to_csv(
    output_folder / "test_target_identities.csv",
    index=False
)


# 15. Save the part metadata for graph construction
part_metadata = (
    panel.loc[
        panel["Entry_Date"] == dates[0],
        ["Reference", "Subsystem"]
    ]
    .set_index("Reference")
    .loc[references]
    .reset_index()
)

part_metadata.to_csv(
    output_folder / "part_metadata.csv",
    index=False
)


# 16. Save the arrays in a compressed NumPy archive
np.savez_compressed(
    output_folder / "forecasting_windows.npz",
    X_train=X_train,
    y_train=y_train,
    y_train_raw=y_train_raw,
    X_val=X_val,
    y_val=y_val,
    y_val_raw=y_val_raw,
    X_test=X_test,
    y_test=y_test,
    y_test_raw=y_test_raw,
    training_origins=training_origins,
    validation_origins=validation_origins,
    test_origins=test_origins,
    references=np.asarray(references, dtype=str),
    feature_names=np.asarray(feature_names, dtype=str),
    target_scale=np.array(target_scale)
)


# 17. Save the preprocessing parameters
metadata = {
    "encoder_length": ENCODER_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "n_parts": n_parts,
    "n_months": n_months,
    "feature_names": feature_names,
    "log1p_features": log_feature_names,
    "feature_scaler_mean": feature_scaler.mean_.tolist(),
    "feature_scaler_scale": feature_scaler.scale_.tolist(),
    "target_scale": target_scale,
    "training_end_date": TRAIN_END_DATE.strftime(
        "%Y-%m-%d"
    ),
    "validation_origin": VALIDATION_ORIGIN_DATE.strftime(
        "%Y-%m-%d"
    ),
    "test_origin": TEST_ORIGIN_DATE.strftime(
        "%Y-%m-%d"
    ),
    "input_array_axes": [
        "forecast_origins",
        "historical_months",
        "parts",
        "features"
    ],
    "target_array_axes": [
        "forecast_origins",
        "parts",
        "future_horizons"
    ]
}

(
    output_folder / "window_preprocessing_metadata.json"
).write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)


# 18. Display the resulting dimensions
print("Training inputs:", X_train.shape)
print("Training targets:", y_train.shape)

print("\nValidation inputs:", X_val.shape)
print("Validation targets:", y_val.shape)

print("\nTest inputs:", X_test.shape)
print("Test targets:", y_test.shape)

print("\nTarget scaling factor:", target_scale)

print("\nValidation and test windows:")
print(
    window_table.loc[
        window_table["Partition"] != "Training"
    ].to_string(index=False)
)

print("\nFirst test target identities:")
print(
    test_target_table.head(12).to_string(index=False)
)

print(f"\nFiles saved in: {output_folder}")